In [ ]:
def IFS_transform(A,beta,c,p):
    n = np.size(A,axis=0)
    d = np.size(A,axis=1)

    mu = np.zeros((n,d))
    nu = np.zeros((n,d))
    pi = np.zeros((n,d))

    for i in range(n):
        for j in range(d):

            z = beta*(A[i][j]-c)

            if z > 50:
                z = 50
            if z < -50:
                z = -50

            mu[i][j] = 1/(1 + np.exp(-z))

            # Yager
            nu[i][j] = (1 - mu[i][j]**p)**(1/p)

            if nu[i][j] > (1 - mu[i][j]):
                nu[i][j] = 1 - mu[i][j] - 1e-8

            pi[i][j] = 1 - mu[i][j] - nu[i][j]

    return mu,nu,pi

In [ ]:
def Calculate_D_ifs(muA,nuA,piA,muV,nuV,piV,i,j):

    term1 = np.linalg.norm(muA[j] - muV[i])**2
    term2 = np.linalg.norm(nuA[j] - nuV[i])**2
    term3 = np.linalg.norm(piA[j] - piV[i])**2

    return term1 + term2 + term3

In [ ]:
def Calculate_SNI(muA,nuA,piA,muV,nuV,piV,U,N,i,j):

    list1 = []
    for r in range(np.size(N,axis=0)):
        if N[r][j] == 1:
            list1.append(r)

    val = 0
    denom = 0

    for k in list1:
        Dik = Calculate_D_ifs(muA,nuA,piA,muV,nuV,piV,i,k)
        val += U[k][i]*Dik
        denom += U[k][i]

    if denom == 0:
        denom = 1e-8

    return val/denom

In [ ]:
def Update_U_ifs(muA,nuA,piA,muV,nuV,piV,m,a_,N):

    n = np.size(muA,axis=0)
    c = np.size(muV,axis=0)

    U = np.zeros((n,c))

    for j in range(n):

        for i in range(c):

            Dij = Calculate_D_ifs(muA,nuA,piA,muV,nuV,piV,i,j)
            Sij = Calculate_SNI(muA,nuA,piA,muV,nuV,piV,U,N,i,j)

            Dtilde = Dij + a_*Sij

            denom = 0

            for k in range(c):
                Dkj = Calculate_D_ifs(muA,nuA,piA,muV,nuV,piV,k,j)
                Skj = Calculate_SNI(muA,nuA,piA,muV,nuV,piV,U,N,k,j)

                Dktilde = Dkj + a_*Skj

                if Dktilde == 0:
                    Dktilde = 1e-8

                denom += (Dtilde/Dktilde)**(1/(m-1))

            U[j][i] = 1/denom

    return U

In [ ]:
def Update_V_ifs(muA,nuA,piA,U,m):

    n = np.size(muA,axis=0)
    c = np.size(U,axis=1)
    d = np.size(muA,axis=1)

    muV = np.zeros((c,d))
    nuV = np.zeros((c,d))
    piV = np.zeros((c,d))

    um = U**m

    for i in range(c):

        denom = np.sum(um[:,i])

        for j in range(n):

            muV[i] += um[j][i]*muA[j]
            nuV[i] += um[j][i]*nuA[j]
            piV[i] += um[j][i]*piA[j]

        muV[i] = muV[i]/denom
        nuV[i] = nuV[i]/denom
        piV[i] = piV[i]/denom

    return muV,nuV,piV

In [ ]:
def Update_B(A,V,U,m):
    um = U**m
    B = A-((um@V)/(um.sum(axis=1)[:,None]))
    return B

In [ ]:
beta = 5
c_param = np.mean(A)
p = 2

muA,nuA,piA = IFS_transform(A,beta,c_param,p)

U_old = U.copy()
V_old = V.copy()
B_old = B.copy()

muV,nuV,piV = IFS_transform(V,beta,c_param,p)

itn = 0

while(itn < 100):

    U_new = Update_U_ifs(muA,nuA,piA,muV,nuV,piV,m,a_,N)

    muV,nuV,piV = Update_V_ifs(muA,nuA,piA,U_new,m)

    B_new = Update_B(A,V,U_new,m)

    if Diff(U_old,U_new,V_old,V,B_old,B_new) < e:
        break

    U_old = U_new.copy()
    B_old = B_new.copy()

    itn = itn + 1